In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Week6_Spark_Assignment") \
    .getOrCreate()
spark

Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.


In [4]:
df = spark.read.csv(
    "/content/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)
df.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

The CSV file was successfully loaded into a Spark DataFrame using header=True to treat the first row as column names and inferSchema=True to automatically detect data types.

Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

In [5]:
df_items = spark.read.csv(
    "/content/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)
df_items.select("product_id", "price").show(5)

+--------------------+-----+
|          product_id|price|
+--------------------+-----+
|4244733e06e7ecb49...| 58.9|
|e5f2d52b802189ee6...|239.9|
|c777355d18b72b67a...|199.0|
|7634da152a4610f15...|12.99|
|ac6c3623068f30de0...|199.9|
+--------------------+-----+
only showing top 5 rows


The required 'category' column was not directly available in the order items dataset. Therefore, additional datasets (products and category translation) were joined using common keys to derive the product category. Filtering was then applied to select only 'Electronics' products along with their corresponding price.

In [6]:
# Alternative way to solve this question using Joins.

df_items = spark.read.csv("/content/olist_order_items_dataset.csv", header=True, inferSchema=True)

df_products = spark.read.csv("/content/olist_products_dataset.csv", header=True, inferSchema=True)

df_cat = spark.read.csv("/content/product_category_name_translation.csv", header=True, inferSchema=True)

df_join = df_items.join(df_products, on="product_id") \
                  .join(df_cat, on="product_category_name")

df_result = df_join.filter(
    df_join["product_category_name_english"] == "electronics"
)

df_result.select("product_id", "price").show(5)

+--------------------+------+
|          product_id| price|
+--------------------+------+
|50fd2b788dc166edd...|  21.9|
|0b0172eb0fd18479d...| 24.89|
|0b0172eb0fd18479d...| 24.89|
|0b0172eb0fd18479d...| 24.89|
|01cf56cd6138b926a...|179.93|
+--------------------+------+
only showing top 5 rows


Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In [7]:
from pyspark.sql.functions import col

df = spark.read.csv(
    "/content/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

df_updated = df.withColumnRenamed("old_name", "new_name") \
               .withColumn("price", col("price").cast("double"))

df_updated.printSchema()
df_updated.show(5)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48

The transformation demonstrates how to rename a column and change the data type of another column in a Spark DataFrame. The column name 'old_name' is used as a placeholder as per the question, since it is not present in the dataset.

Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [8]:
df_orders = spark.read.csv("/content/olist_orders_dataset.csv", header=True, inferSchema=True)
df_items = spark.read.csv("/content/olist_order_items_dataset.csv", header=True, inferSchema=True)

df_joined = df_orders.join(df_items, on="order_id")

df_filtered = df_joined.filter(
    (df_joined["order_status"] == "delivered") &
    (df_joined["price"] > 1000)
)

df_filtered.select("order_id", "order_status", "price").show(5)

+--------------------+------------+------+
|            order_id|order_status| price|
+--------------------+------------+------+
|403b97836b0c04a62...|   delivered|1299.0|
|2b72a32a2c3e93fa2...|   delivered|1148.0|
|ac64f79a33bfa575f...|   delivered|1099.0|
|6cb134bb285a64b04...|   delivered|1999.0|
|da8be3bb62e9bf01e...|   delivered|2690.0|
+--------------------+------------+------+
only showing top 5 rows


Since the 'amount' column was not available in the orders dataset, the 'price' column from the order items dataset was used by performing a join on 'order_id'. This allowed complete implementation of the filtering condition using available data.

Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [9]:
from pyspark.sql.functions import col

df_items = spark.read.csv(
    "/content/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

from pyspark.sql.functions import round

df_final = df_items.withColumn(
    "final_price",
    round(col("price") * 1.18, 2)
)

df_final.select("product_id", "price", "final_price").show(5)

+--------------------+-----+-----------+
|          product_id|price|final_price|
+--------------------+-----+-----------+
|4244733e06e7ecb49...| 58.9|       69.5|
|e5f2d52b802189ee6...|239.9|     283.08|
|c777355d18b72b67a...|199.0|     234.82|
|7634da152a4610f15...|12.99|      15.33|
|ac6c3623068f30de0...|199.9|     235.88|
+--------------------+-----+-----------+
only showing top 5 rows


he dataset does not contain a column named 'base_price'. Therefore, the existing 'price' column was used as a substitute to calculate the 'final_price' by applying an 18% tax.

Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [ ]:
df = spark.read.parquet("path/to/input")

df_filtered = df.filter(df["user_id"].isNotNull())

df_filtered.write.csv("path/to/output", header=True)

The file paths used are placeholders as per the question. In a real-world scenario, they should be replaced with actual file paths. The code demonstrates loading a Parquet file, filtering null values, and saving the result in CSV format.

Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [10]:
from pyspark.sql.functions import when

df_orders = spark.read.csv("/content/olist_orders_dataset.csv", header=True, inferSchema=True)

df_customers = spark.read.csv("/content/olist_customers_dataset.csv", header=True, inferSchema=True)

df_items = spark.read.csv("/content/olist_order_items_dataset.csv", header=True, inferSchema=True)

df_joined = df_orders.join(df_customers, on="customer_id") \
                     .join(df_items, on="order_id")

df_joined = df_joined.withColumn(
    "priority",
    when(df_joined["price"] > 1000, "High").otherwise("Low")
)

df_filtered = df_joined.filter(
    (df_joined["customer_state"] == "SP") | (df_joined["priority"] == "High")
)

df_filtered.select("order_id", "customer_state", "priority", "price").show(5)

+--------------------+--------------+--------+-----+
|            order_id|customer_state|priority|price|
+--------------------+--------------+--------+-----+
|432aaf21d85167c2c...|            SP|     Low|38.25|
|203096f03d82e0dff...|            SP|     Low|109.9|
|f848643eec1d69395...|            SP|     Low|79.99|
|f3e7c359154d96582...|            SP|     Low| 89.9|
|dd78f560c270f1909...|            SP|     Low| 47.9|
+--------------------+--------------+--------+-----+
only showing top 5 rows


Since the dataset does not contain explicit 'region' and 'priority' columns, these were derived using multiple datasets. The 'region' was approximated using the customer state from the customers dataset, and 'priority' was created based on price conditions. The datasets were joined using common keys to implement the required filtering logic.